# Mission d'adaptation : un nouveau site, sans code fourni
**ÉNONCÉ**

Vous savez maintenant scraper et crawler un site (Books to Scrape), et le faire poliment. **Cette mission change de site, et on ne vous donne aucun code de collecte.** C'est à vous d'adapter votre démarche à une structure différente.

**Le site :** `quotes.toscrape.com`, des citations célèbres, créé lui aussi pour s'entraîner.

## Ce que vous devez produire
1. **`citations`** : un tableau de **toutes** les citations (toutes les pages), avec les colonnes `texte`, `auteur`, `tags`, `url_page`, `url_auteur`.
2. **`auteurs`** : un tableau de la fiche de **chaque auteur, une seule fois**, avec `url_auteur`, `auteur`, `date_naissance`, `lieu_naissance`, `biographie`.
3. **`nb_requetes`** : le nombre de pages que votre script a demandées (pages de citations + fiches d'auteurs, sans compter `robots.txt`).
4. **`citations_completes`** : les citations enrichies de la fiche de leur auteur.
5. **`top_auteurs`** et **`top_tags`** : les 5 auteurs les plus cités et les 10 tags les plus fréquents.
6. **Une note de 5 lignes** : ce qui change par rapport à Books to Scrape, et comment vous l'avez géré.

## Règles
Les règles du notebook précédent s'appliquent : vous vous **présentez** (User-Agent avec contact), vous **lisez le robots.txt**, vous faites une **pause**. Ces outils vous sont fournis plus bas.

## Pour vous mettre sur la piste (sans code)
- **Inspectez** le site dans votre navigateur avant d'écrire quoi que ce soit. Qu'est-ce qui est pareil ? Qu'est-ce qui est différent ?
- Regardez **la forme des adresses** des liens « Next » et « (about) ».
- Un **même auteur** apparaît dans de nombreuses citations. Faut-il télécharger sa fiche à chaque fois ?
- Une citation a **plusieurs tags**. Comment stocker cela dans une seule case ?
- Le texte d'une citation est entouré de **guillemets typographiques**. Faut-il les garder ?

In [1]:
import re, time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from urllib.robotparser import RobotFileParser

SITE = "https://quotes.toscrape.com"      # un NOUVEAU site-école, à la structure différente de Books to Scrape

def controle(condition, ok, ko):
    print("✅", ok) if condition else print("❌", ko)

## Étape 1 — Se présenter
Comme dans le notebook précédent : remplacez `___` par votre contact (adresse e-mail ou adresse d'une page de projet).

In [2]:
ENTETES = {"User-Agent": "TPFormation/1.0 (citation@test.fr) python-requests"}   # TODO : remplacez ___ par votre contact
print(ENTETES)

{'User-Agent': 'TPFormation/1.0 (citation@test.fr) python-requests'}


In [3]:
# Contrôle (ne pas modifier)
ua = ENTETES['User-Agent']
controle(('@' in ua or 'http' in ua) and '___' not in ua and not any(m in ua.lower() for m in ('example', 'exemple', 'votre.')), 'User-Agent renseigné : ' + ua, 'Remplacez ___ par un vrai contact')

✅ User-Agent renseigné : TPFormation/1.0 (citation@test.fr) python-requests


## Étape 2 — Vos outils (fournis)
La fonction de requête polie et la lecture du `robots.txt`, issues du notebook précédent.

In [4]:
PAUSE = 1.0

def get_poli(url, essais=3):
    for n in range(essais):
        time.sleep(PAUSE)
        reponse = requests.get(url, headers=ENTETES, timeout=20)
        if reponse.status_code in (429, 503):
            time.sleep(2 ** (n + 1))
            continue
        return reponse
    raise RuntimeError("Échec après " + str(essais) + " essais : " + url)

def lire_robots(base):
    reponse = requests.get(base + "/robots.txt", headers=ENTETES)
    print(base, "-> code", reponse.status_code)
    if reponse.status_code == 404:
        print("   pas de fichier robots.txt : aucune restriction annoncée")
        return None
    if reponse.status_code != 200:
        raise RuntimeError("Réponse inattendue pour robots.txt : on s'abstient par prudence")
    regles = RobotFileParser(); regles.parse(reponse.text.splitlines())
    print(reponse.text[:500])
    return regles

## Étape 3 — Consulter le robots.txt
Lisez le `robots.txt` de ce site, puis demandez s'il est permis de visiter la page 2 des citations (`/page/2/`) et la fiche d'un auteur (`/author/Albert-Einstein`). Stockez les deux réponses (vrai ou faux) dans `autorise_pages` et `autorise_auteur`.

In [6]:
regles = lire_robots(SITE)
# TODO : calculer autorise_pages et autorise_auteur (si regles vaut None, aucune restriction annoncée)
autorise_pages = True if regles is None else regles.can_fetch(ua, SITE + "/page/2/")
autorise_auteur = True if regles is None else regles.can_fetch(ua, SITE + "/author/Albert-Einstein")

print("Pages :", autorise_pages)
print("Auteur :", autorise_auteur)

https://quotes.toscrape.com -> code 404
   pas de fichier robots.txt : aucune restriction annoncée
Pages : True
Auteur : True


In [7]:
# Contrôle (ne pas modifier)
controle(isinstance(autorise_pages, bool) and isinstance(autorise_auteur, bool), 'Autorisations établies : pages=%s, auteur=%s' % (autorise_pages, autorise_auteur), 'Attendu : deux booléens')

✅ Autorisations établies : pages=True, auteur=True


## Étape 4 — Collecter toutes les citations *(à écrire seul)*
Objectif : la variable **`citations`** (un `DataFrame`) et le compteur **`nb_requetes`**. Cette cellule est **vide** : à vous d'écrire la boucle.

In [11]:
reponse = requests.get(SITE)
print("Code de réponse :", reponse.status_code)

page = BeautifulSoup(reponse.content, "html.parser")
print("Titre de la page :", page.title.get_text(strip=True))

carte = page.select_one("div.quote")
print(carte.prettify()[:900])

Code de réponse : 200
Titre de la page : Quotes to Scrape
<div class="quote" itemscope="" itemtype="http://schema.org/CreativeWork">
 <span class="text" itemprop="text">
  “The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”
 </span>
 <span>
  by
  <small class="author" itemprop="author">
   Albert Einstein
  </small>
  <a href="/author/Albert-Einstein">
   (about)
  </a>
 </span>
 <div class="tags">
  Tags:
  <meta class="keywords" content="change,deep-thoughts,thinking,world" itemprop="keywords"/>
  <a class="tag" href="/tag/change/page/1/">
   change
  </a>
  <a class="tag" href="/tag/deep-thoughts/page/1/">
   deep-thoughts
  </a>
  <a class="tag" href="/tag/thinking/page/1/">
   thinking
  </a>
  <a class="tag" href="/tag/world/page/1/">
   world
  </a>
 </div>
</div>



In [21]:
def lire_citation(carte, url_page):
    lien = carte.select_one('a[href^="/author/"]')
    texte = carte.select_one(".text").get_text(strip=True)
    texte = texte.strip('“”"')
    tags = [tag.get_text(strip=True) for tag in carte.select(".tag")]
    return {
        "auteur": carte.select_one(".author").get_text(strip=True),
        "tags": tags,
        "texte": texte,
        "url_auteur": urljoin(url_page, lien["href"]) if lien else None,
        "url_page": url_page
    }

In [22]:
# À vous : collecter toutes les citations (variables attendues : citations, nb_requetes)
nb_requetes = 0
url = SITE
resultats = []

while url:
    reponse = get_poli(url)
    reponse.raise_for_status()
    nb_requetes += 1

    page = BeautifulSoup(reponse.content, "html.parser")

    for carte in page.select("div.quote"):
        resultats.append(lire_citation(carte, url))

    suivant = page.select_one("li.next a")
    url = urljoin(url, suivant["href"]) if suivant else None

citations = pd.DataFrame(resultats)

print(f"{len(citations)} citations récupérées")
print(f"{nb_requetes} requêtes")
display(citations.head())


100 citations récupérées
10 requêtes


,auteur,tags,texte,url_auteur,url_page
0,Albert Einstein,"[change, deep-thoughts, thinking, world]",The world as we have created it is a process o...,https://quotes.toscrape.com/author/Albert-Eins...,https://quotes.toscrape.com
1,J.K. Rowling,"[abilities, choices]","It is our choices, Harry, that show what we tr...",https://quotes.toscrape.com/author/J-K-Rowling,https://quotes.toscrape.com
2,Albert Einstein,"[inspirational, life, live, miracle, miracles]",There are only two ways to live your life. One...,https://quotes.toscrape.com/author/Albert-Eins...,https://quotes.toscrape.com
3,Jane Austen,"[aliteracy, books, classic, humor]","The person, be it gentleman or lady, who has n...",https://quotes.toscrape.com/author/Jane-Austen,https://quotes.toscrape.com
4,Marilyn Monroe,"[be-yourself, inspirational]","Imperfection is beauty, madness is genius and ...",https://quotes.toscrape.com/author/Marilyn-Monroe,https://quotes.toscrape.com


In [23]:
# Contrôle (ne pas modifier)
colonnes = {"texte", "auteur", "tags", "url_page", "url_auteur"}
controle(isinstance(citations, pd.DataFrame) and colonnes <= set(citations.columns) and len(citations) >= 10,
         "%d citations collectées, colonnes correctes" % len(citations), "Attendu : un DataFrame avec les colonnes " + ", ".join(sorted(colonnes)) + " et au moins 10 lignes")
controle(citations["tags"].map(lambda t: isinstance(t, list)).all() and not citations["texte"].str.startswith(("“", '"')).any(),
         "Les tags sont des listes et les guillemets typographiques sont retirés", "Les tags doivent être des listes, et le texte ne doit plus commencer par un guillemet")
controle(citations["url_auteur"].str.startswith("http").all() and citations["url_page"].str.startswith("http").all(),
         "Toutes les adresses sont complètes", "Les adresses doivent commencer par http : pensez aux liens relatifs")
controle(not citations.duplicated(subset=["texte", "auteur"]).any(), "Aucune citation en double", "Il y a des citations en double : une page a-t-elle été lue deux fois ?")

✅ 100 citations collectées, colonnes correctes
✅ Les tags sont des listes et les guillemets typographiques sont retirés
✅ Toutes les adresses sont complètes
✅ Aucune citation en double


## Étape 5 — Collecter la fiche de chaque auteur *(à écrire seul)*
Objectif : la variable **`auteurs`**, avec **une seule ligne par auteur**. Inspectez une fiche d'auteur pour trouver où sont la date et le lieu de naissance, et la biographie. **Attention au nombre de requêtes** : le contrôle suivant le vérifie.

In [25]:
# À vous : collecter une fiche par auteur (variable attendue : auteurs)
def lire_auteur(page, url):
    nom = page.select_one(".author-title")
    date = page.select_one(".author-born-date")
    lieu = page.select_one(".author-born-location")
    bio = page.select_one(".author-description")
    return {
        "auteur": nom.get_text(strip=True) if nom else None,
        "date_naissance": date.get_text(strip=True) if date else None,
        "lieu_naissance": lieu.get_text(strip=True) if lieu else None,
        "biographie": bio.get_text(" ", strip=True) if bio else None,
        "url_auteur": url
    }

urls_auteurs = citations["url_auteur"].dropna().unique()

resultats_auteurs = []

for url in urls_auteurs:
    reponse = get_poli(url)
    reponse.raise_for_status()

    page = BeautifulSoup(reponse.content, "html.parser")
    resultats_auteurs.append(lire_auteur(page, url))

auteurs = pd.DataFrame(resultats_auteurs)

print(f"{len(auteurs)} auteurs uniques récupérés")
display(auteurs.head())

50 auteurs uniques récupérés


,auteur,date_naissance,lieu_naissance,biographie,url_auteur
0,Albert Einstein,"March 14, 1879","in Ulm, Germany","In 1879, Albert Einstein was born in Ulm, Germ...",https://quotes.toscrape.com/author/Albert-Eins...
1,J.K. Rowling,"July 31, 1965","in Yate, South Gloucestershire, England, The U...",See also: Robert GalbraithAlthough she writes ...,https://quotes.toscrape.com/author/J-K-Rowling
2,Jane Austen,"December 16, 1775","in Steventon Rectory, Hampshire, The United Ki...",Jane Austen was an English novelist whose work...,https://quotes.toscrape.com/author/Jane-Austen
3,Marilyn Monroe,"June 01, 1926",in The United States,Marilyn Monroe (born Norma Jeane Mortenson; Ju...,https://quotes.toscrape.com/author/Marilyn-Monroe
4,André Gide,"November 22, 1869","in Paris, France",André Paul Guillaume Gide was a French author ...,https://quotes.toscrape.com/author/Andre-Gide


In [26]:
# Contrôle (ne pas modifier)
colonnes_a = {"url_auteur", "auteur", "date_naissance", "lieu_naissance", "biographie"}
controle(isinstance(auteurs, pd.DataFrame) and colonnes_a <= set(auteurs.columns) and auteurs["url_auteur"].is_unique,
         "%d fiches d'auteurs, une seule par auteur" % len(auteurs), "Attendu : les colonnes " + ", ".join(sorted(colonnes_a)) + " et une seule ligne par auteur")
controle(set(citations["url_auteur"]) <= set(auteurs["url_auteur"]), "Tous les auteurs cités ont une fiche", "Certains auteurs cités n'ont pas de fiche")
attendu = citations["url_page"].nunique() + citations["url_auteur"].nunique()
controle(nb_requetes <= attendu + 2, "%d requêtes pour %d attendues au minimum : fiches non redemandées" % (nb_requetes, attendu), "%d requêtes alors que %d suffisent : un auteur a été redemandé plusieurs fois" % (nb_requetes, attendu))

✅ 50 fiches d'auteurs, une seule par auteur
✅ Tous les auteurs cités ont une fiche
✅ 10 requêtes pour 60 attendues au minimum : fiches non redemandées


## Étape 6 — Fusionner et analyser *(à écrire seul)*
Objectifs : **`citations_completes`** (chaque citation avec la fiche de son auteur), **`top_auteurs`** (les 5 auteurs les plus cités) et **`top_tags`** (les 10 tags les plus fréquents). Indice : pour compter des tags stockés dans des listes, il faut d'abord « déplier » ces listes.

In [27]:
# À vous : citations_completes, top_auteurs, top_tags

citations_completes = citations.merge(
    auteurs,
    on="url_auteur",
    how="left",
    suffixes=("", "_fiche")
)

top_auteurs = citations["auteur"].value_counts().head(5)

top_tags = citations.explode("tags")["tags"].value_counts().head(10)

print("Citations complètes :")
display(citations_completes.head())

print("Top 5 des auteurs :")
display(top_auteurs)

print("Top 10 des tags :")
display(top_tags)

Citations complètes :


,auteur,tags,texte,url_auteur,url_page,auteur_fiche,date_naissance,lieu_naissance,biographie
0,Albert Einstein,"[change, deep-thoughts, thinking, world]",The world as we have created it is a process o...,https://quotes.toscrape.com/author/Albert-Eins...,https://quotes.toscrape.com,Albert Einstein,"March 14, 1879","in Ulm, Germany","In 1879, Albert Einstein was born in Ulm, Germ..."
1,J.K. Rowling,"[abilities, choices]","It is our choices, Harry, that show what we tr...",https://quotes.toscrape.com/author/J-K-Rowling,https://quotes.toscrape.com,J.K. Rowling,"July 31, 1965","in Yate, South Gloucestershire, England, The U...",See also: Robert GalbraithAlthough she writes ...
2,Albert Einstein,"[inspirational, life, live, miracle, miracles]",There are only two ways to live your life. One...,https://quotes.toscrape.com/author/Albert-Eins...,https://quotes.toscrape.com,Albert Einstein,"March 14, 1879","in Ulm, Germany","In 1879, Albert Einstein was born in Ulm, Germ..."
3,Jane Austen,"[aliteracy, books, classic, humor]","The person, be it gentleman or lady, who has n...",https://quotes.toscrape.com/author/Jane-Austen,https://quotes.toscrape.com,Jane Austen,"December 16, 1775","in Steventon Rectory, Hampshire, The United Ki...",Jane Austen was an English novelist whose work...
4,Marilyn Monroe,"[be-yourself, inspirational]","Imperfection is beauty, madness is genius and ...",https://quotes.toscrape.com/author/Marilyn-Monroe,https://quotes.toscrape.com,Marilyn Monroe,"June 01, 1926",in The United States,Marilyn Monroe (born Norma Jeane Mortenson; Ju...


Top 5 des auteurs :


auteur
Albert Einstein    10
J.K. Rowling        9
Marilyn Monroe      7
Dr. Seuss           6
Mark Twain          6
Name: count, dtype: int64

Top 10 des tags :


tags
love             14
inspirational    13
life             13
humor            12
books            11
reading           7
friendship        5
friends           4
truth             4
simile            3
Name: count, dtype: int64

In [28]:
# Contrôle (ne pas modifier)
controle(len(citations_completes) == len(citations) and "date_naissance" in citations_completes.columns and citations_completes["date_naissance"].notna().all(),
         "Fusion correcte : chaque citation a la fiche de son auteur", "La fusion doit garder toutes les citations, chacune avec la date de naissance de son auteur")
controle(len(top_auteurs) == min(5, citations["auteur"].nunique()) and len(top_tags) == min(10, citations.explode("tags")["tags"].nunique()) and top_auteurs.is_monotonic_decreasing and top_tags.is_monotonic_decreasing,
         "Classements corrects : 5 auteurs et 10 tags, du plus fréquent au moins fréquent", "Attendu : 5 auteurs et 10 tags triés par fréquence décroissante")

✅ Fusion correcte : chaque citation a la fiche de son auteur
✅ Classements corrects : 5 auteurs et 10 tags, du plus fréquent au moins fréquent


## Étape 7 — Votre note (5 lignes)
Répondez ici, en 5 lignes maximum :

1. Ce qui était **identique** à Books to Scrape.
2. Ce qui était **différent**.
3. Ce que vous avez dû **changer** dans votre démarche.
4. Comment vous avez évité de **redemander** les fiches d'auteurs.
5. Ce qui, à votre avis, **casserait** votre script si le site changeait.

Points en communs: L'utilisation de Requests, BeautifulSoup et pandas. Avec une boucle pour parcourir les pages et extraire les données.   
Différences: Les données et la structure HTML changent, notamment les citations, leurs tages et les fiches auteurs.     
Adaptation: J'ai du modifier les sélecteurs CSS, nettoyer les textes, gérer les listes de tags et fusionner les données. 
Doublons: J'ai utilisé unique() sur les url des auteurs pour éviter de télécharger plusieurs fois les mêmes fiches.  
Risques: Si les classes HTML, les liens ou le système de pagination changent, mon script risque de ne plus trouver les informations.  
> 